# Subset sensitivity

In [1]:
import pandas as pd
import numpy as np
from monte import train_with_cv

## Load data

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_train = pd.read_parquet("../../data/methylation/train_val_pan-cancer_beta.parquet")
df_meta_train = pd.read_csv("../../data/methylation/train_val_pan-cancer_meta.csv")
df_meta_train = df_meta_train.set_index("Barcode", drop=False)
df_meta_train = df_meta_train.loc[df_beta_train.index]

we used the metric as the primary metric for learning the cancer purity

In [3]:
metric = "CPE"
df_meta_metric = df_meta_train.dropna(subset=[metric])
df_beta_metric = df_beta_train.loc[df_meta_metric["Barcode"]]

**Test set**

In [ ]:
# the parquet file and metadata csv file could be downloaded from Zenodo
df_beta_test = pd.read_parquet("../../data/methylation/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/methylation/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

## Model training with subset

In [5]:
subset_sizes = [50, 100, 500, 1000, 5000, 10000, 50000, 100000]
n_repeats = 10

In [6]:
def evaluate_predictions(y_true, y_pred):
    from sklearn.metrics import mean_squared_error

    mse = mean_squared_error(y_true, y_pred)
    r2 = np.corrcoef(y_true, y_pred)[0, 1]

    return {"MSE": mse, "correlation": r2}

df_results = []
for subset_size in subset_sizes:
    for repeat in range(n_repeats):
        np.random.seed(repeat)
        subset_probes = np.random.choice(df_beta_metric.columns, size=subset_size, replace=False)
        df_beta_subset = df_beta_metric[subset_probes]

        monte = train_with_cv(df_beta_subset, df_meta_metric[metric])

        # predict on training and test sets
        pred_train = monte.predict_purity(df_beta_subset)
        pred_test = monte.predict_purity(df_beta_test[subset_probes])

        # evaluation
        eval_train = evaluate_predictions(df_meta_metric[metric], pred_train)
        eval_test = evaluate_predictions(df_meta_test[metric], pred_test)
        df_results.append({
            "subset_size": subset_size,
            "repeat": repeat,
            "train_MSE": eval_train["MSE"],
            "train_correlation": eval_train["correlation"],
            "test_MSE": eval_test["MSE"],
            "test_correlation": eval_test["correlation"]
        })

df_results = pd.DataFrame(df_results)

In [9]:
df_results

,subset_size,repeat,train_MSE,train_correlation,test_MSE,test_correlation
0,50,0,0.067638,0.333466,0.062707,0.337032
1,50,1,0.030891,0.574725,0.030093,0.581919
2,50,2,0.029830,0.609053,0.028209,0.630125
3,50,3,0.051850,0.382153,0.046519,0.412541
4,50,4,0.046103,0.492378,0.045040,0.499939
...,...,...,...,...,...,...
75,100000,5,0.008530,0.846517,0.007704,0.864145
76,100000,6,0.008457,0.847967,0.007631,0.865189
77,100000,7,0.008450,0.848046,0.007666,0.864991
78,100000,8,0.008483,0.847524,0.007621,0.865373


In [10]:
df_results.to_csv("../../data/monte_outputs/pancancer/monte_pancancer_model_probe_subset.csv", index=False)